# 🛡️ PhishGuard — Explainable Phishing Website Detection

**Title:** Explainable Phishing Website Detection Using Behavioral and Structural Features  
**Notebook:** Experimentation & Walkthrough  

> ⚠️ **Disclaimer:** Model predictions are estimates, not guarantees of website safety.

---

## What This Notebook Does

This notebook walks through the complete PhishGuard pipeline interactively:

1. Setup and imports  
2. Dataset loading and inspection  
3. Data preprocessing  
4. URL feature extraction  
5. Model training and comparison  
6. Model evaluation  
7. SHAP explanations  
8. LIME explanations  
9. Live URL prediction  

**For Google Colab users:** Run each cell top-to-bottom. Install dependencies in the first cell.

## 1. Setup — Install Dependencies & Configure Paths

In [ ]:
# ─── Install dependencies (run this in Google Colab or fresh environments) ───
# Comment this out if you already have packages installed.

import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

packages = [
    "pandas", "numpy", "scikit-learn",
    "xgboost", "lightgbm",
    "shap", "lime",
    "matplotlib", "seaborn", "joblib",
]

for p in packages:
    install(p)

print("✓ All packages installed.")

In [ ]:
# ─── Configure project path ───
# If running in Google Colab, update this path to where you uploaded the project.
# If running locally, leave as-is (assumes notebook is in the project root).

import os, sys

# Change this if you are running from Google Colab:
# PROJECT_ROOT = '/content/drive/MyDrive/phishguard'
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))  # project root

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

import config
print(f"Data directory: {config.DATA_DIR}")
print(f"Models directory: {config.MODELS_DIR}")

In [ ]:
# ─── Standard imports ───
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.facecolor'] = '#1e293b'
plt.rcParams['figure.facecolor'] = '#0f172a'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

print("✓ Imports ready.")

---
## 2. Dataset Loading & Inspection

**Before running this section:**  
Download the Hannousse & Yahiouche (2021) dataset from:  
https://data.mendeley.com/datasets/c2gw7fy2j4/3  
Place the CSV file at: `data/dataset_phishing.csv`

In [ ]:
from src.data_loader import load_primary_dataset, inspect_dataset, verify_primary_dataset

# Load the dataset
# If the file is not found, you'll see clear instructions.
df = load_primary_dataset()

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)[:10]}...")  # Show first 10 columns

In [ ]:
# Inspect dataset — shows missing values, duplicates, label distribution
info = inspect_dataset(df)

# Verify it matches expected schema
verify_result = verify_primary_dataset(df)

In [ ]:
# Plot class distribution
target_col = config.PRIMARY_TARGET_COLUMN

if target_col in df.columns:
    label_counts = df[target_col].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Bar chart
    colors = ['#43A047', '#E53935']
    label_counts.plot(kind='bar', color=colors, ax=ax1)
    ax1.set_title('Class Distribution', fontsize=14, color='white')
    ax1.set_ylabel('Count')
    ax1.tick_params(axis='x', rotation=0)
    
    # Pie chart
    ax2.pie(label_counts.values, labels=label_counts.index,
            colors=colors, autopct='%1.1f%%', startangle=90)
    ax2.set_title('Class Proportions', fontsize=14, color='white')
    
    plt.suptitle('Dataset Label Distribution', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"Target column '{target_col}' not found. Check config.py.")

In [ ]:
# Preview the first few rows
df.head(3)

---
## 3. Data Preprocessing

In [ ]:
from src.preprocessing import preprocess_dataset

# This single function does everything:
# 1. Encodes labels (legitimate=0, phishing=1)
# 2. Removes duplicate rows
# 3. Splits into train/val/test (70/15/15)
# 4. Fits imputer on training data ONLY
# 5. Transforms all splits

data = preprocess_dataset(df)

print(f"\n✓ Preprocessing complete!")
print(f"  X_train shape: {data['X_train'].shape}")
print(f"  X_val shape:   {data['X_val'].shape}")
print(f"  X_test shape:  {data['X_test'].shape}")
print(f"  Features: {len(data['feature_names'])}")

---
## 4. URL Feature Extraction

This module extracts structural features from a URL string **without visiting the website**.

In [ ]:
from src.feature_extraction import extract_url_features, get_feature_names, compare_with_dataset_features

# Extract features from example URLs
test_urls = [
    "https://www.google.com",
    "http://192.168.1.1/login",
    "https://paypal-secure-login.tk/verify?account=123",
    "https://bit.ly/fake-login",
]

print("URL Feature Extraction Demo:\n")
for url in test_urls:
    try:
        feats = extract_url_features(url)
        print(f"URL: {url}")
        print(f"  URL length:       {feats['url_length']}")
        print(f"  Subdomains:       {feats['num_subdomains']}")
        print(f"  HTTPS:            {feats['has_https']}")
        print(f"  IP address:       {feats['has_ip_address']}")
        print(f"  URL shortener:    {feats['is_shortened']}")
        print(f"  Suspicious TLD:   {feats['has_suspicious_tld']}")
        print()
    except ValueError as e:
        print(f"  Error: {e}\n")

In [ ]:
# Show full feature schema
feature_names = get_feature_names()
print(f"Total URL-extractable features: {len(feature_names)}")
print("\nFeature list:")
for i, name in enumerate(feature_names, 1):
    print(f"  {i:2}. {name}")

In [ ]:
# Compare URL features with dataset features
# This shows which dataset features can/cannot be reproduced from a URL
report = compare_with_dataset_features(data['feature_names'])

print(f"\nShared features ({report['n_shared']}): {report['shared']}")
print(f"\nDataset-only features ({len(report['in_dataset_only'])}) — cannot be extracted from URL alone:")
for f in report['in_dataset_only'][:10]:
    print(f"  - {f}")
if len(report['in_dataset_only']) > 10:
    print(f"  ... and {len(report['in_dataset_only'])-10} more")

---
## 5. Model Training

In [ ]:
from src.train import train_all_models, save_models

# Train all 4 candidate models
# This may take a few minutes depending on your hardware
print("Training models... (this may take 2-5 minutes)\n")

results = train_all_models(data)

# Save models and pipeline to disk
save_models(results, data)

In [ ]:
# Plot validation performance comparison
valid_results = {k: v for k, v in results.items() if not v.get('skipped')}

if valid_results:
    models_list = list(valid_results.keys())
    val_acc = [valid_results[m]['val_accuracy'] for m in models_list]
    val_f1  = [valid_results[m]['val_f1'] for m in models_list]

    x = range(len(models_list))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar([i - width/2 for i in x], val_acc, width, label='Validation Accuracy', color='#3b82f6')
    ax.bar([i + width/2 for i in x], val_f1, width, label='Validation F1', color='#f59e0b')
    ax.set_xticks(list(x))
    ax.set_xticklabels(models_list)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Validation Set Performance Comparison', fontsize=14)
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 6. Model Evaluation (Test Set)

In [ ]:
from src.evaluate import evaluate_model
import joblib

# Evaluate the best model on the held-out test set
# We use Random Forest as the baseline
model_name = 'RandomForest'
model_path = config.SAVED_MODEL_TEMPLATE.format(model_name=model_name)

if os.path.isfile(model_path):
    model = joblib.load(model_path)
    label_encoder = joblib.load(config.SAVED_LABEL_ENCODER_PATH)
    
    test_result = evaluate_model(
        model, data['X_test'], data['y_test'],
        model_name, label_encoder
    )
else:
    print(f"Model not found: {model_path}. Run training first.")

In [ ]:
# Confusion matrix
if 'test_result' in dir():
    from sklearn.metrics import ConfusionMatrixDisplay
    import numpy as np
    
    cm = np.array(test_result['confusion_matrix'])
    target_names = list(label_encoder.classes_)
    
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {model_name} (Test Set)', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Note what the cells mean:
    print("\nInterpreting the confusion matrix:")
    print(f"  True Negatives  (legitimate → legitimate): {cm[0,0]}")
    print(f"  False Positives (legitimate → phishing):   {cm[0,1]}  ← Legitimate sites wrongly flagged")
    print(f"  False Negatives (phishing → legitimate):   {cm[1,0]}  ← DANGEROUS: phishing missed!")
    print(f"  True Positives  (phishing → phishing):     {cm[1,1]}")

In [ ]:
# ROC Curve
if 'test_result' in dir() and 'y_proba' in test_result:
    from sklearn.metrics import roc_curve
    
    fpr, tpr, _ = roc_curve(data['y_test'], test_result['y_proba'])
    auc = test_result['roc_auc']
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, color='#3b82f6', linewidth=2,
            label=f'{model_name} (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random baseline')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve (Test Set)', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

---
## 7. SHAP Explanations

SHAP (SHapley Additive exPlanations) shows which features the model relies on most.

> **Important:** SHAP values explain model behavior, not proof of malicious intent.

In [ ]:
from src.explain import explain_shap_global

if 'model' in dir():
    print("Computing SHAP global feature importance...")
    print("(May take 30–60 seconds for large datasets)\n")
    
    shap_result = explain_shap_global(
        model, data['X_test'],
        data['feature_names'],
        model_name=model_name,
        max_samples=300,
    )
    
    if 'error' in shap_result:
        print(f"SHAP error: {shap_result['error']}")
    else:
        if shap_result.get('figure'):
            plt.figure(shap_result['figure'].number)
            plt.show()
        
        print("\nTop 10 most important features:")
        print(shap_result['feature_importance'].head(10).to_string(index=False))
else:
    print("Load a model first (run the training section above).")

In [ ]:
# SHAP local explanation — for a single test sample
from src.explain import explain_shap_local, generate_plain_english_explanation

if 'model' in dir() and 'shap_result' in dir() and 'error' not in shap_result:
    # Pick the first test sample
    single_sample = data['X_test'][[0]]
    actual_label_idx = data['y_test'][0]
    actual_label = label_encoder.inverse_transform([actual_label_idx])[0]
    
    print(f"Sample #0 — Actual label: {actual_label}")
    
    local_result = explain_shap_local(
        model, single_sample,
        data['feature_names'],
        model_name=model_name,
    )
    
    if 'error' in local_result:
        print(f"SHAP local error: {local_result['error']}")
    else:
        if local_result.get('figure'):
            plt.figure(local_result['figure'].number)
            plt.show()
        
        # Plain-English explanation
        pred_label = model.predict(single_sample)
        pred_str = label_encoder.inverse_transform(pred_label)[0]
        explanation = generate_plain_english_explanation(
            local_result['contributions'], pred_str
        )
        print("\n" + explanation)

---
## 8. LIME Explanations

LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions by creating a local surrogate model.

In [ ]:
from src.explain import explain_lime

if 'model' in dir():
    single_sample = data['X_test'][[0]]
    
    print("Computing LIME explanation...")
    lime_result = explain_lime(
        model, single_sample,
        data['feature_names'],
        training_data=data['X_train'],
        model_name=model_name,
        num_features=15,
    )
    
    if 'error' in lime_result:
        print(f"LIME error: {lime_result['error']}")
    else:
        if lime_result.get('figure'):
            plt.figure(lime_result['figure'].number)
            plt.show()
        
        print("\nTop feature weights from LIME:")
        print(lime_result['feature_weights'].head(10).to_string(index=False))
else:
    print("Load a model first.")

---
## 9. Live URL Prediction

In [ ]:
# First, train the URL-only model if not already done
from src.train import train_url_only_model

print("Training URL-only model...")
url_model_result = train_url_only_model(df)

if url_model_result.get('model') is None:
    print("⚠ URL-only model could not be trained. See message above.")
else:
    print(f"✓ URL-only model trained.")
    print(f"  Validation accuracy: {url_model_result.get('val_accuracy', 'N/A')}")

In [ ]:
from src.predict import PhishingPredictor

predictor = PhishingPredictor()

# Test URLs — these are examples only, not endorsements
test_urls = [
    "https://www.wikipedia.org",
    "http://192.168.1.1/admin/login",
    "https://paypal-secure-verification.tk/login?user=victim",
    "https://github.com/features",
]

print("LIVE URL PREDICTIONS")
print("=" * 60)
print()

for url in test_urls:
    result = predictor.predict(url)
    
    if result['error']:
        print(f"URL: {url}")
        print(f"  ❌ Error: {result['error']}")
    else:
        icon = '⚠️' if result['prediction'] == 'phishing' else '✓'
        confidence = f"{result['confidence']:.1%}" if result['confidence'] else 'N/A'
        phish_prob = f"{result['phishing_probability']:.1%}" if result['phishing_probability'] else 'N/A'
        
        print(f"{icon} URL: {url}")
        print(f"  Prediction:          {result['prediction'].upper()}")
        print(f"  Model Confidence:    {confidence}")
        print(f"  Phishing Probability:{phish_prob}")
    print()

print("─" * 60)
print("⚠ REMINDER: These predictions are model estimates.")
print("  'legitimate' does NOT mean the website is safe.")

---
## 10. Summary & Next Steps

### What we built:
- ✅ Loaded and inspected the phishing detection dataset
- ✅ Preprocessed data with train/val/test split (no leakage)
- ✅ Extracted 28 structural URL features
- ✅ Trained 4 ML models and compared them
- ✅ Evaluated on held-out test set
- ✅ Generated SHAP and LIME explanations
- ✅ Demonstrated live URL prediction

### To launch the Streamlit dashboard:
```bash
streamlit run app.py
```

### To run automated tests:
```bash
python -m pytest tests/ -v
```

### Limitations to be aware of:
1. The URL-only model uses 28 features vs. 87 in the full dataset
2. Model confidence ≠ actual real-world safety
3. Novel phishing techniques may evade the model
4. This is an educational project, not a production security tool